# Empirical Analysis of Exchange Rate Dynamics: A Macroeconomic Framework

## 1. Theoretical Note on Exchange Rates

In open-economy macroeconomics, the exchange rate is a pivotal variable that links domestic and foreign markets, influencing both the real economy and financial flows. It serves as the primary transmission mechanism for exogenous monetary and fiscal shocks across sovereign borders.

### Nominal vs. Real Exchange Rates
The **Nominal Exchange Rate** ($S$) is defined as the relative price of two sovereign currencies. For the purposes of this note, we adopt the standard European quotation convention (indirect quotation): the number of units of foreign currency required to purchase one unit of domestic currency (e.g., USD/EUR). Under this convention, an increase in $S$ signifies an appreciation of the domestic currency.

To evaluate true international competitiveness, financial engineers and macroeconomists rely on the **Real Exchange Rate** ($Q$). This metric adjusts the nominal rate for differential price levels between the domestic and foreign economies:

$Q = S \cdot \frac{P}{P^*}$

Where:
* **$Q$**: Real Exchange Rate.
* **$S$**: Nominal Exchange Rate.
* **$P$**: Domestic price level (e.g., GDP Deflator or CPI).
* **$P^*$**: Foreign price level.

### Macroeconomic Significance and Net Exports
The real exchange rate directly dictates the trajectory of the Net Exports ($NX$) component of the Gross Domestic Product ($Y = C + I + G + NX$). Assuming the Marshall-Lerner condition holds, Net Exports are a decreasing function of the real exchange rate:

$NX = NX(Q, Y, Y^*)$

A depreciation in the real exchange rate (a decline in $Q$) stimulates foreign demand for domestic exports ($X$) and suppresses domestic demand for foreign imports ($M$), holding domestic ($Y$) and foreign ($Y^*$) income constant. 

### Historical Volatility
In financial engineering, historical volatility ($\sigma$) is standardly estimated using the sample standard deviation of daily logarithmic returns, scaled by the square root of the number of trading periods in a year (typically 252 for Forex).

$\sigma_t = \sqrt{252} \cdot \sqrt{\frac{1}{n-1} \sum_{i=0}^{n-1} \left( r_{t-i} - \bar{r}_t \right)^2}$

In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

%load_ext sql
%sql duckdb:///:memory:

Connecting to 'duckdb:///:memory:'

In [2]:
%%sql
INSTALL httpfs;
LOAD httpfs;
INSTALL json;
LOAD json;

Running query in 'duckdb:///:memory:'

Success


In [3]:
%%sql processed_fx <<

WITH raw_usd AS (
    -- Sostituisci l'URL qui sotto con l'endpoint corretto per i tassi storici (es. dailyRates o timeSeries)
    SELECT 
        CAST(json_extract(unnested_rates, '$.referenceDate') AS DATE) AS date,
        CAST(json_extract(unnested_rates, '$.avgRate') AS DOUBLE) AS eur_usd
    FROM (
        -- Assicurati che l'array contenente i tassi nel nuovo JSON si chiami effettivamente 'rates'
        SELECT unnest(rates) AS unnested_rates 
        FROM read_json_auto(https://tassidicambio.bancaditalia.it/terzevalute-wf-web/rest/v1.0/dailyTimeSeries?startDate=2022-01-01&endDate=2024-01-01&currencyIsoCode=USD)
    )
    WHERE eur_usd IS NOT NULL
),
fx_returns AS (
    -- Calcola i log rendimenti
    SELECT 
        date,
        eur_usd,
        LN(eur_usd) - LN(LAG(eur_usd) OVER (ORDER BY date)) AS log_return
    FROM raw_usd
)
-- Calcola la volatilità storica a 30 giorni, annualizzata
SELECT 
    date,
    eur_usd,
    log_return,
    STDDEV_SAMP(log_return) OVER (
        ORDER BY date 
        ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ) * SQRT(252) AS rolling_annualized_volatility
FROM fx_returns
ORDER BY date ASC;

Running query in 'duckdb:///:memory:'

RuntimeError: If using snippets, you may pass the --with argument explicitly.
For more details please refer: https://jupysql.ploomber.io/en/latest/compose.html#with-argument


Original error message from DB driver:
(_duckdb.ParserException) Parser Error: syntax error at or near ":"

LINE 9:         FROM read_json_auto(https://tassidicambio.bancaditalia.it/terzevalute-wf-web/rest...
                                         ^
[SQL: WITH raw_usd AS (

    SELECT
        CAST(json_extract(unnested_rates, '$.referenceDate') AS DATE) AS date,
        CAST(json_extract(unnested_rates, '$.avgRate') AS DOUBLE) AS eur_usd
    FROM (

        SELECT unnest(rates) AS unnested_rates
        FROM read_json_auto(https://tassidicambio.bancaditalia.it/terzevalute-wf-web/rest/v1.0/dailyTimeSeries?startDate=2022-01-01&endDate=2024-01-01&currencyIsoCode=USD)
    )
    WHERE eur_usd IS NOT NULL
),
fx_returns AS (

    SELECT
        date,
        eur_usd,
        LN(eur_usd) - LN(LAG(eur_usd) OVER (ORDER B

## 2. Statistical Metrics

We transition the in-memory DuckDB relational construct into a Pandas DataFrame to leverage its advanced descriptive statistical methods, providing a quantitative baseline for the stochastic behavior of the asset.

In [ ]:
# Convert DuckDB ResultSet to Pandas DataFrame
df_fx = processed_fx.DataFrame()
df_fx['date'] = pd.to_datetime(df_fx['date'])
df_fx.set_index('date', inplace=True)

# Display summary statistics
print("--- Empirical Statistical Metrics (EUR/USD) ---")
display(df_fx.describe().round(4))

## 3. Empirical Visualization of Trajectories

Visualizing the dual temporal evolution of the nominal exchange rate and its localized (rolling) volatility. This dual-axis approach allows the financial engineer to identify distinct macroeconomic regimes, such as volatility clustering during central bank interventions or global liquidity shocks.

In [ ]:
# Filter out the initial NaNs from the rolling window computation
df_plot = df_fx.dropna()

fig, ax1 = plt.subplots(figsize=(14, 7))

# Primary Axis: Nominal Exchange Rate
color1 = 'midnightblue'
ax1.set_xlabel('Date', fontweight='bold')
ax1.set_ylabel('Nominal Exchange Rate (USD per 1 EUR)', color=color1, fontweight='bold')
line1 = ax1.plot(df_plot.index, df_plot['eur_usd'], color=color1, linewidth=2, label='EUR/USD Spot Rate')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(True, linestyle='--', alpha=0.5)

# Secondary Axis: Rolling Annualized Volatility
ax2 = ax1.twinx()  
color2 = 'crimson'
ax2.set_ylabel('Annualized Volatility (30-Day Rolling)', color=color2, fontweight='bold')  
line2 = ax2.plot(df_plot.index, df_plot['rolling_annualized_volatility'], color=color2, linewidth=1.5, alpha=0.8, label='30D Volatility')
ax2.tick_params(axis='y', labelcolor=color2)

# Formatting the X-axis for better date readability
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Combine legends from both axes
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper right', frameon=True, shadow=True)

plt.title('EUR/USD: Nominal Rate Trajectory vs. Stochastic Volatility\n(Banca d\'Italia API | DuckDB Memory Execution)', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()